<a href="https://colab.research.google.com/github/mxls34/AdvanceDatabase/blob/main/Ch7_%E0%B9%81%E0%B8%9A%E0%B8%9A%E0%B8%9D%E0%B8%B6%E0%B8%81%E0%B8%AB%E0%B8%B1%E0%B8%94.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

รหัสนักศึกษา 6706021612143 ชื่อ-สกุล นาย ณัฐกรณ์ มะลิซ้อน

# Chapter 7 - แบบฝึกหัด

**Domain:** ระบบจองที่นั่งคอนเสิร์ต (Concert Seat Booking)

แบบฝึกหัดนี้มี **8 ข้อ**

**บริบท:** งานคอนเสิร์ตมีที่นั่งจำกัด 5 ที่นั่ง ลูกค้าหลายคนอาจพยายามจองที่นั่งเดียวกัน "พร้อมกัน" ผ่านแอประบบจองตั๋ว


## Setup — เตรียมฐานข้อมูล

In [1]:
!pip install psycopg2-binary -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 31.6 MB/s eta 0:00:00


ใช้ connection string เดียวกับที่เก็บไว้ใน Colab Secrets ชื่อ `NEON_CONNECTION_STRING` (เหมือนใน Lab)

In [2]:
import psycopg2
from google.colab import userdata

CONN_STRING = userdata.get('NEON_CONNECTION_STRING')

def get_conn(autocommit=False):
    conn = psycopg2.connect(CONN_STRING)
    conn.autocommit = autocommit
    return conn

def run(conn, sql, params=None, fetch=True):
    cur = conn.cursor()
    cur.execute(sql, params)
    if fetch and cur.description:
        rows = cur.fetchall()
        cols = [d[0] for d in cur.description]
        cur.close()
        return cols, rows
    cur.close()
    return None, None

def show(cols, rows):
    if cols is None:
        print("(no result set)")
        return
    print(" | ".join(cols))
    for r in rows:
        print(" | ".join(str(v) for v in r))

print("เชื่อมต่อ Neon สำเร็จ")

เชื่อมต่อ Neon สำเร็จ


In [3]:
setup_conn = get_conn(autocommit=True)
run(setup_conn, """
DROP TABLE IF EXISTS seats;
CREATE TABLE seats (
    seat_id     INT PRIMARY KEY,
    seat_label  TEXT NOT NULL,
    status      TEXT NOT NULL DEFAULT 'available' CHECK (status IN ('available', 'booked')),
    booked_by   TEXT
);
""", fetch=False)

run(setup_conn, """
INSERT INTO seats (seat_id, seat_label, status) VALUES
    (1, 'A1', 'available'),
    (2, 'A2', 'available'),
    (3, 'A3', 'available'),
    (4, 'A4', 'available'),
    (5, 'A5', 'available');
""", fetch=False)

cols, rows = run(setup_conn, "SELECT * FROM seats ORDER BY seat_id;")
show(cols, rows)
setup_conn.close()

seat_id | seat_label | status | booked_by
1 | A1 | available | None
2 | A2 | available | None
3 | A3 | available | None
4 | A4 | available | None
5 | A5 | available | None


---
## ข้อ 1

อธิบายว่า **Transaction** คืออะไร และคุณสมบัติ **ACID ข้อใดข้อหนึ่ง** ที่เกี่ยวข้องโดยตรงที่สุดกับการป้องกัน "ที่นั่งเดียวถูกจองซ้ำโดยสองคน" พร้อมอธิบายเหตุผล

> ✍️ เขียนคำตอบของคุณในเซลล์นี้

_(คำตอบ)_

Transaction คือ
หน่วยของงาน ที่กระทำในเวลานั้นๆ ประกอบไปด้วยชุดคําสั่งสําหรับ อ่าน หรือ เขียนข้อมูล

C = Consistency ข้อมูลต้องถูกต้องตามกฎเสมอ

**เหตุผล**

เมื่อที่นั่ง A ถูกจองเรียบร้อยแล้ว สถานะ ควรขึ้นว่า unavailable  และไม่สามารถถูกจ้องที่นั่ง A ซ้ำได้

---
## ข้อ 2

เขียนคำสั่งจองที่นั่ง A5 (seat_id = 5) ให้ลูกค้าชื่อ "Nueng" แบบที่ **ปลอดภัยจาก Race Condition** ในคำสั่งเดียว (ไม่ต้อง SELECT ก่อน UPDATE) โดยเงื่อนไขคือ ต้องจองสำเร็จ **เฉพาะกรณีที่นั่งยัง available เท่านั้น**

In [4]:
booking_conn = get_conn(autocommit=True)
cur = booking_conn.cursor()
cur.execute(
    "UPDATE seats SET status ='booked', booked_by=%s WHERE seat_id=%s AND status ='available';",
    ('Nueng', 5)
)
if cur.rowcount > 0:
    print("จองสำเร็จ!")
else:
    print("จองไม่สำเร็จ — ที่นั่งนี้ถูกจองไปแล้ว")
cur.close()
booking_conn.close()


จองสำเร็จ!


**ทำไมวิธีนี้ปลอดภัยกว่าการ `SELECT` เช็คสถานะก่อน แล้วค่อย `UPDATE` ทีหลัง?**
(คำสั่ง UPDATE เดี่ยวๆ ของ PostgreSQL เป็น atomic operation ในตัวเอง — ไม่มีช่องว่างระหว่างการอ่านกับการเขียนให้ transaction อื่นแทรกได้)

---
## ข้อ 3

สมมติระบบเขียนโค้ดจองที่นั่งแบบนี้แทน (แยก SELECT กับ UPDATE เป็นสองคำสั่ง):

```python
# ขั้นที่ 1: เช็คสถานะ
cur.execute("SELECT status FROM seats WHERE seat_id = 5;")
status = cur.fetchone()[0]

# ขั้นที่ 2: ถ้าว่าง ก็จอง
if status == 'available':
    cur.execute("UPDATE seats SET status='booked', booked_by=%s WHERE seat_id=5;", (name,))
```

อธิบายว่าโค้ดนี้เสี่ยงเกิดปัญหาอะไร (ระบุชื่อปัญหาที่ตรงกับที่เรียนในทฤษฎี) หากมีลูกค้าสองคนกดจองที่นั่งเดียวกัน "พร้อมกัน" และจะเกิดผลเสียอะไรตามมาในทางธุรกิจ

> ✍️ เขียนคำตอบของคุณในเซลล์นี้

_(คำตอบ)_

การ SELECT เพื่อตรวจสอบสถานะก่อน แล้วจึง UPADTE ในภายหลัง แบบแยกคำสั่ง เสี่ยงต่อการเกิดปัญหา lost update เช่น ลูกค้าคนที่ 1 และคนที่ 2 จองที่นั่งเดียวกัน อาจทำให้ข้อมูลของคนใดคนหนึ่ง ถูกเขียนทับ และเกิดข้อมูลหาย

---
## ข้อ 4

จำลองสถานการณ์ในข้อ 3 ด้วยที่นั่ง A1 (seat_id = 1): เปิด 2 transaction (T1 = ลูกค้า "Fa", T2 = ลูกค้า "Gift") ให้ทั้งคู่ **`SELECT` เช็คสถานะก่อน แล้วค่อย `UPDATE`** แบบแยกคำสั่ง (เหมือนโค้ดในข้อ 3) แล้วดูว่าใครเป็นคนได้ที่นั่งไปในที่สุด

In [5]:
conn_t1 = get_conn()
conn_t2 = get_conn()

# Start transactions for T1 and T2
run(conn_t1, "BEGIN;", fetch=False)
run(conn_t2, "BEGIN;", fetch=False)

# T1 and T2 both SELECT the status
_, rows_t1 = run(conn_t1, "SELECT status FROM seats WHERE seat_id = 1;")
_, rows_t2 = run(conn_t2, "SELECT status FROM seats WHERE seat_id = 1;")
status_t1, status_t2 = rows_t1[0][0], rows_t2[0][0]
print("T1 เห็นสถานะ:", status_t1, " | T2 เห็นสถานะ:", status_t2)

# T1 attempts to book
if status_t1 == 'available':
    run(conn_t1, "UPDATE seats SET status='booked', booked_by='Fa' WHERE seat_id=1;", fetch=False)
    run(conn_t1, "COMMIT;", fetch=False)
    print("T1 (Fa): จองสำเร็จตามตรรกะของตัวเอง")

# T2 attempts to book
if status_t2 == 'available':
    run(conn_t2, "UPDATE seats SET status='booked', booked_by='Gift' WHERE seat_id=1;", fetch=False)
    run(conn_t2, "COMMIT;", fetch=False)
    print("T2 (Gift): จองสำเร็จตามตรรกะของตัวเอง")

conn_t1.close(); conn_t2.close()

check_conn = get_conn(autocommit=True)
cols, rows = run(check_conn, "SELECT * FROM seats WHERE seat_id = 1;")
show(cols, rows)
check_conn.close()
print("\nสังเกต: ทั้ง Fa และ Gift ต่างคิดว่าตัวเอง 'จองสำเร็จ' แต่ในฐานข้อมูลมีชื่อเดียวที่ถูกบันทึกไว้จริง")

T1 เห็นสถานะ: available  | T2 เห็นสถานะ: available
T1 (Fa): จองสำเร็จตามตรรกะของตัวเอง
T2 (Gift): จองสำเร็จตามตรรกะของตัวเอง
seat_id | seat_label | status | booked_by
1 | A1 | booked | Gift

สังเกต: ทั้ง Fa และ Gift ต่างคิดว่าตัวเอง 'จองสำเร็จ' แต่ในฐานข้อมูลมีชื่อเดียวที่ถูกบันทึกไว้จริง


---
## ข้อ 5

รีเซ็ตสถานะที่นั่ง A1 กลับเป็น `available` แล้วแก้โค้ดในข้อ 4 ให้ปลอดภัยด้วย **`SELECT ... FOR UPDATE`** (ให้ T1 ล็อกก่อน T2 ค่อยพยายามอ่าน) เพื่อพิสูจน์ว่า T2 จะไม่มีทางเห็นสถานะ `available` ผิดๆ อีกต่อไป

In [6]:
reset_conn = get_conn(autocommit=True)
run(reset_conn, "UPDATE seats SET status='available', booked_by=NULL WHERE seat_id=1;", fetch=False)
reset_conn.close()
print("รีเซ็ตที่นั่ง A1 กลับเป็น available แล้ว")

รีเซ็ตที่นั่ง A1 กลับเป็น available แล้ว


In [7]:
conn_t1 = get_conn()
run(conn_t1, "BEGIN;", fetch=False)
_, rows_t1 = run(conn_t1, "SELECT status FROM seats WHERE seat_id = 1 FOR UPDATE;")
if rows_t1[0][0] == 'available':
    run(conn_t1, "UPDATE seats SET status='booked', booked_by='Fa' WHERE seat_id=1;", fetch=False)
run(conn_t1, "COMMIT;", fetch=False)
print("T1 (Fa): ล็อกแถว จองสำเร็จ แล้ว COMMIT ปล่อยล็อก")
conn_t1.close()

conn_t2 = get_conn()
run(conn_t2, "BEGIN;", fetch=False)
_, rows_t2 = run(conn_t2, "SELECT status FROM seats WHERE seat_id = 1 FOR UPDATE;")
print("T2 เห็นสถานะ (หลัง T1 commit แล้ว):", rows_t2[0][0])
if rows_t2[0][0] == 'available':
    run(conn_t2, "UPDATE seats SET status='booked', booked_by='Gift' WHERE seat_id=1;", fetch=False)
    print("T2 (Gift): จองสำเร็จ")
else:
    print("T2 (Gift): เห็นว่าถูกจองไปแล้ว — ไม่จอง")
run(conn_t2, "COMMIT;", fetch=False)
conn_t2.close()

check_conn = get_conn(autocommit=True)
cols, rows = run(check_conn, "SELECT * FROM seats WHERE seat_id = 1;")
show(cols, rows)
check_conn.close()

T1 (Fa): ล็อกแถว จองสำเร็จ แล้ว COMMIT ปล่อยล็อก
T2 เห็นสถานะ (หลัง T1 commit แล้ว): booked
T2 (Gift): เห็นว่าถูกจองไปแล้ว — ไม่จอง
seat_id | seat_label | status | booked_by
1 | A1 | booked | Fa


**สังเกตผล:** ด้วย `FOR UPDATE`, T2 จะต้อง **รอ** จนกว่า T1 จะ COMMIT ก่อนเสมอ แล้วจึงเห็นสถานะล่าสุด (`booked` ไม่ใช่ `available` อีกต่อไป) — ป้องกัน Lost Update / Overbooking ได้สำเร็จ

---
## ข้อ 6

ถ้าไม่ใช้ `FOR UPDATE` แต่จะแก้ปัญหาในข้อ 3–4 ด้วยการ **เปลี่ยน Isolation Level** แทน ควรเลือกระดับใด (`READ COMMITTED`, `REPEATABLE READ`, หรือ `SERIALIZABLE`) จึงจะป้องกัน Lost Update ได้แน่นอน 100% พร้อมอธิบายเหตุผล และข้อแลกเปลี่ยน (trade-off) ที่ต้องยอมรับ

> ✍️ เขียนคำตอบของคุณในเซลล์นี้

_(คำตอบ)_


SERIALIZABLE เป็นการบังคับให้ทุกการทำธุรกรรม (Transaction) ทำงานทีละรายการ ไม่พร้อมกันเลยแม้แต่น้อย Ex. ถ้ามีคนหลายคนพยายามจองที่นั่งเดียวกัน ระบบจะจัดคิวให้ทีละคนเท่านั้น ทำให้คนที่จองก่อนได้ข้อมูลที่ถูกต้องและจองสำเร็จไปก่อน ส่วนคนถัดมาก็จะเห็นสถานะที่อัปเดตแล้วทันทีว่าที่นั่งไม่ว่างแล้ว จึงไม่มีทางที่ข้อมูลจะผิดพลาดหรือจองทับซ้อนกันได้เลย ป้องกันปัญหาจองซ้ำหรือข้อมูลหายได้ 100% แต่ระบบจะทำงานช้าลงมาก

---
## ข้อ 7

ถ้าระบบอนุญาตให้ลูกค้าจอง **สองที่นั่งพร้อมกันในคำสั่งเดียว** (เช่น จองคู่ A1+A2) โดยใช้ `SELECT ... FOR UPDATE` ล็อกทีละที่นั่งตามลำดับที่ลูกค้าเลือก อธิบายว่าอาจเกิด **Deadlock** ขึ้นได้อย่างไร พร้อมเสนอวิธีป้องกันเชิงออกแบบ

> ✍️ เขียนคำตอบของคุณในเซลล์นี้

_(คำตอบ)_

**Deadlock** เกิดเมื่อลูกค้าสองคนจอง 2 ที่นั่ง (เช่น A1 A2) พร้อมกัน ต่างคนต่างล็อกที่นั่งคนละที่ก่อน แล้วพยายามล็อกที่นั่งที่อีกฝ่ายล็อกอยู่ ทำให้เกิดการรอซึ่งกันและกัน **วิธีป้องกัน** ควรกำหนดให้ล็อกที่นั่งตามลำดับที่แน่นอนเสมอ (เช่น เรียงตาม `seat_id`) เพื่อป้องกันการรอแบบวงกลม และทำให้การทำธุรกรรมดำเนินไปได้อย่างราบรื่น

---
## ข้อ 8

ระบบจองตั๋วคอนเสิร์ตดังในโจทย์นี้ มักมีช่วงเวลาที่คนเข้าใช้พร้อมกันจำนวนมากมาก (เช่น ตอนเปิดขายตั๋วรอบแรก) — เปรียบเทียบข้อดี/ข้อเสียของการใช้ **Locking (Pessimistic)** ล้วนๆ กับการพึ่งพา **MVCC** เป็นหลัก (แล้วจัดการ conflict ตอน commit) สำหรับสถานการณ์นี้โดยเฉพาะ

> ✍️ เขียนคำตอบของคุณในเซลล์นี้

_(คำตอบ)_

สำหรับระบบจองตั๋วคอนเสิร์ตที่มีผู้ใช้จำนวนมากช่วงเปิดขายตั๋ว **Locking (Pessimistic)** ปลอดภัยสูง แต่ลด Concurrency อย่างมาก ทำให้ระบบช้าและเสี่ยง Deadlock ส่วน **MVCC** รองรับ Concurrency ได้ดีกว่ามาก ระบบลื่นไหล ตอบสนองเร็วกว่า แต่ต้องจัดการ Conflict ตอน Commit ด้วยการ Rollback/Retry ในแอปพลิเคชัน ดังนั้น MVCC มักเป็นทางเลือกที่ดีกว่า เพราะเน้น High Concurrency ที่สำคัญในช่วงเวลาที่มีผู้ใช้จำนวนมหาศาล